In [ ]:
# Copyright 2026 Arm Limited and/or its affiliates.
#
# This source code is licensed under the BSD-style license found in the
# LICENSE file in the root directory of this source tree.

**NOTICE:** *MXFP is not yet fully supported in the VGF backend. This example instead uses the TOSA reference model until VGF has full support.*

# Running MXFP on Arm backend

This guide demonstrates the full flow for quantizing select submodules to MXFP on the Arm backend.

Before you begin:
1. (In a clean virtual environment with a compatible Python version) Install executorch using `./install_executorch.sh`
2. Install Arm cross-compilation toolchain and simulators using `./examples/arm/setup.sh --i-agree-to-the-contained-eula`
3. Export vulkan environment variables and add MLSDK components to PATH and LD_LIBRARY_PATH using `examples/arm/arm-scratch/setup_path.sh`

With all commands executed from the base `executorch` folder.

### Define the module and quantize it to MXFP

The following code block shows how to use the `to_mxfp` API to quantize a PyTorch module to an MXFP representation. We start by defining the `torch.nn.Module` we want to quantize, then an `MXFPOpConfig` specifiying how to perform the quantization is created. The config selects which datatype to quantize to, and which exact layers to target. `to_mxfp` then quantizes the module in place.

In [ ]:
import torch
from torch import nn

from executorch.backends.arm.ao_ext import MXFPOpConfig, to_mxfp

torch.manual_seed(0)

def filter_only_fc1(mod: nn.Module, name: str) -> bool:
    return name == "fc1"        


class Mod(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(32, 64)
        self.fc2 = nn.Linear(64, 16)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x


module = Mod().eval()
print(f"Initial module:\n{module}\n")

# F8E4M3 is used here. See backends/arm/ao_ext/mxfp.py for all supported datatypes.
config = MXFPOpConfig(
    weight_dtype=torch.float8_e4m3fn,
)
# To quantize only module.fc1, add back the commented out filter_fn arg.
to_mxfp(
    module,
    config,
    # filter_fn=filter_only_fc1,
)
print(f"MXFP-quantized module:\n{module}\n")


example_inputs = (torch.randn(1, 32),)
exported_program = torch.export.export(module, example_inputs)
graph_module = exported_program.module(check_guards=False)
print("Exported module:")
_ = graph_module.print_readable()

In [ ]:
import os
from executorch.backends.arm.tosa.partitioner import TOSAPartitioner
from executorch.backends.arm.tosa.compile_spec import TosaCompileSpec
from executorch.exir import (
    EdgeCompileConfig,
    ExecutorchBackendConfig,
    to_edge_transform_and_lower,
)
from executorch.extension.export_util.utils import save_pte_program


# TODO MLETORCH-2141: MXFP is not fully supported in the VGF toolchain yet.
# Use the TOSA reference model in the mean time and switch to VgfPartitioner
# when full support is in place.
compile_spec = TosaCompileSpec("TOSA-1.1+FP+mxfp")
partitioner = TOSAPartitioner(compile_spec)

# Lower the exported program to the TOSA backend
edge_program_manager = to_edge_transform_and_lower(
    exported_program,
    partitioner=[partitioner],
    compile_config=EdgeCompileConfig(
        _check_ir_validity=False,
    ),
)

# Convert edge program to executorch
executorch_program_manager = edge_program_manager.to_executorch(
    config=ExecutorchBackendConfig(extract_delegate_segments=False)
)
executorch_program_manager.exported_program().module(check_guards=False).print_readable()

# Save pte file
cwd_dir = os.getcwd()
pte_base_name = "mxfp_example"
pte_name = pte_base_name + ".pte"
pte_path = os.path.join(cwd_dir, pte_name)
save_pte_program(executorch_program_manager, pte_name)
assert os.path.exists(pte_path), "Build failed; no .pte-file found"

In [ ]:
from executorch.backends.arm.test.runner_utils import TosaReferenceModelDispatch

# TODO MLETORCH-2141: Run on VGF backend instead when it's supported.
with torch.no_grad():
    lowered_module = executorch_program_manager.exported_program().graph_module
    with TosaReferenceModelDispatch():
        tosa_ref_output = lowered_module(*example_inputs)

print("Model output:")
print(tosa_ref_output)


### Porting over already quantized modules

The flow presented in this notebook shows how to quantize a module to MXFP, but what if we have a module that was quantized outside of the Arm backend's `to_mxfp` flow? Then we need to port this module to the representation that is compatible with the Arm backend. Doing this requires manual work, but it is possible; note that `to_mxfp` simply replaces submodules with corresponding MXFP implementations, e.g., `torch.nn.Linear` is replaced with `executorch.backends.arm.ao_ext.ops.MXFPLinearOp`. With this observation we can replicate this procedure manually by reassigning the module's weights. The code snippet below shows how to replace the model `Mod`'s `self.fc1` with some made-up pretrained MXFP4 weights we could have gotten from somewhere else.

In [ ]:
import torch
from executorch.backends.arm.ao_ext.ops import MXFPLinearOp

block_size = 32
in_features = 32
out_features = 64

# Note: float4_e2m1fn_x2 weights are packed pairwise into uint8 arrays
example_external_w_data = torch.randint(
    0,
    256,
    (1, out_features, in_features // 2),
    dtype=torch.uint8,
)
example_external_w_scale = torch.randint(
    0,
    256,
    (1, out_features, in_features // block_size),
    dtype=torch.float8_e8m0fnu,
)
example_external_bias = torch.randn(out_features, dtype=torch.float32)

module = Mod().eval()
module.fc1 = MXFPLinearOp(
    example_external_w_data,
    example_external_w_scale,
    example_external_bias,
    torch.float4_e2m1fn_x2,
    block_size,
)

port_output = module(example_inputs[0])
assert port_output.shape == (1, 16)
print(module)
